# **Import Lib**

In [4]:
import requests
import torch
from PIL import Image
from transformers import *
from tqdm import tqdm
import pickle
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import DataParallel
from torch.optim import AdamW
import urllib.parse as parse
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print("The device used is", device)

The device used is cuda


**Ẩn log**

In [5]:
import transformers
transformers.logging.set_verbosity_error()

In [6]:
# from google.colab import drive
import os
import torch
import torchvision.datasets as dset
import torchvision.transforms as transforms

# Xử lý Data
**Cần đổi đường dẫn data**

In [7]:
import json
##### load data caption
train_file = "/kaggle/input/datasets/nikhil7280/coco-image-caption/annotations_trainval2014/annotations/captions_train2014.json"
val_file = "/kaggle/input/datasets/nikhil7280/coco-image-caption/annotations_trainval2017/annotations/captions_val2017.json"
with open(train_file, "r") as f:
    train_coco = json.load(f)
with open(val_file, "r") as f:
    val_coco = json.load(f)
train_id2file = {}
val_id2file = {}
for img in train_coco["images"]:
    train_id2file[img["id"]] = img["file_name"]
for img in val_coco["images"]:
    val_id2file[img["id"]] = img["file_name"]


**Mapping Ảnh - caption**

**1 ảnh - 5caption**

In [8]:
from PIL import Image
from tqdm import tqdm

train_dir = "/kaggle/input/datasets/nikhil7280/coco-image-caption/train2014/train2014"
val_dir = "/kaggle/input/datasets/nikhil7280/coco-image-caption/val2017/val2017"
train_image2sentences = {}
val_image2sentences = {}
for ann in train_coco["annotations"]:
    image_id = ann["image_id"]

    if image_id not in train_image2sentences:
        train_image2sentences[image_id] = []

    train_image2sentences[image_id].append({
        "raw": ann["caption"]
    })

for ann in val_coco["annotations"]:
    image_id = ann["image_id"]

    if image_id not in val_image2sentences:
        val_image2sentences[image_id] = []

    val_image2sentences[image_id].append({
        "raw": ann["caption"]
    })

**Train, Valid, Test**

**Dạng 1 path - 1 caption**

**File test giữ nguyên 1 path - 5 caps**

In [9]:
train_samples = []
val_samples = []
test_samples = []
for image_id, sentences in tqdm(train_image2sentences.items()):
    for cap in sentences:

        train_samples.append({
            "image_path": f"{train_dir}/{train_id2file[image_id]}",
            "caption": cap["raw"]
        })
idx = 0
for image_id, sentences in tqdm(val_image2sentences.items()):
    if idx <= 0.9* len(val_image2sentences):
        for cap in sentences:
    
            val_samples.append({
                "image_path": f"{val_dir}/{val_id2file[image_id]}",
                "caption": cap["raw"]
            })
    else:
        test_samples.append({
                "image_path": f"{val_dir}/{val_id2file[image_id]}",
                "caption": [cap["raw"] for cap in sentences]
            })
    idx+=1

100%|██████████| 5000/5000 [00:00<00:00, 191722.08it/s]


In [10]:
test_samples[1]

{'image_path': '/kaggle/input/datasets/nikhil7280/coco-image-caption/val2017/val2017/000000161861.jpg',
 'caption': ['A woman standing on a tennis court holding a racquet.',
  'A female tennis player is posed and ready for the ball to come.',
  'THERE IS A WOMAN THAT IS HOLDNG A TENNIS RACKET ',
  'A young woman playing tennis on a grass tennis court.',
  'A woman tennis player getting ready to be served the ball.']}

In [11]:
from datasets import Dataset

train_ds_coco = Dataset.from_list(train_samples)
val_ds_coco = Dataset.from_list(val_samples)

In [12]:
import pickle

with open("train_ds_coco.pkl", "wb") as f:
    pickle.dump(train_ds_coco, f)

with open("val_ds_coco.pkl", "wb") as f:
    pickle.dump(val_ds_coco, f)


In [13]:
# a function to determine whether a string is a URL or not
def is_url(string):
    try:
        result = parse.urlparse(string)
        return all([result.scheme, result.netloc, result.path])
    except:
        return False

In [14]:
def load_image(image_path):
    if is_url(image_path):
        return Image.open(image_path).convert("RGB")   # 🔥 FIX
    elif os.path.exists(image_path):
        return Image.open(image_path).convert("RGB")

# **Load Model**

In [ ]:
import os
# the encoder model
encoder_model = "microsoft/swin-base-patch4-window7-224-in22k"
# the decoder model 
decoder_model = "gpt2"
decoder_config = AutoConfig.from_pretrained(decoder_model)
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    encoder_model, decoder_model, decoder_config=decoder_config, output_hidden_states = True
).to(device)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from transformers import (
    
    GPT2TokenizerFast,
    ViTImageProcessor
)

tokenizer = GPT2TokenizerFast.from_pretrained(decoder_model)
image_processor = ViTImageProcessor.from_pretrained(encoder_model)

max_length = 32

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

In [ ]:
if "gpt2" in decoder_model:
  tokenizer.pad_token = tokenizer.eos_token
  model.config.eos_token_id = tokenizer.eos_token_id
  model.config.pad_token_id = tokenizer.pad_token_id
  model.config.decoder_start_token_id = tokenizer.bos_token_id
else:
  model.config.decoder_start_token_id = tokenizer.cls_token_id
  model.config.pad_token_id = tokenizer.pad_token_id
import pickle
with open('train_ds_coco.pkl', 'rb') as f:
    train_ds = pickle.load(f)
    
with open('val_ds_coco.pkl', 'rb') as f:
    valid_ds = pickle.load(f)

**Xử lý thành dạng 'pixel_values'-'labels(ids)'** 

In [ ]:
def preprocess(items):
  # preprocess the image
  images   = []
  captions = []

  imgs      = items["image_path"]     if isinstance(items["image_path"],     list) else [items["image_path"]]
  sentences = items["caption"] if isinstance(items["caption"], list) else [items["caption"]]

  for img, sents in zip(imgs, sentences):
            images.append(load_image(img))
            captions.append(sents)
  pixel_values = image_processor(images, return_tensors="pt").pixel_values.to(device)
  # tokenize the caption with truncation and padding
  targets = tokenizer(captions,
                      max_length=max_length, padding="max_length", truncation=True, return_tensors="pt").to(device)
  return {'pixel_values': pixel_values, 'labels': targets["input_ids"]}


train_dataset = train_ds.with_transform(preprocess)
valid_dataset = valid_ds.with_transform(preprocess)

In [ ]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch])
    }
    

# **Tham số **
**chỉnh tham số ở đây**

In [20]:
num_epochs = 15 # number of epochs
batch_size = 8 # the size of batches


#Dataloader for data to be fed to model
from torch.utils.data import DataLoader

# define our data loaders
train_dataset_loader = DataLoader(train_dataset, collate_fn=collate_fn, batch_size=batch_size, shuffle=True)
valid_dataset_loader = DataLoader(valid_dataset, collate_fn=collate_fn, batch_size=2, shuffle=True)

n_train_steps = num_epochs * len(train_dataset_loader)
n_valid_steps = len(valid_dataset_loader)
current_step = 0
save_steps = 5000
optimizer = AdamW(model.parameters(), lr=1e-4)

vocab_size = 50257

# **Đánh giá BLEU1_4-CIDER-METEOR**

In [22]:
!pip install pycocoevalcap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 18.0 MB/s eta 0:00:0000:0100:01


In [23]:
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor


def compute_caption_metrics(
    predictions: dict[int, list[str]],
    ground_truths: dict[int, list[str]],
    verbose: bool = True
) -> dict[str, float]:
    """
    Tính BLEU-1~4, CIDEr, METEOR cho image captioning.

    Args:
        predictions  : {image_id: ["predicted caption"]}
        ground_truths: {image_id: ["cap1", "cap2", "cap3", "cap4", "cap5"]}
        verbose      : In kết quả ra màn hình

    Returns:
        dict chứa tất cả scores
    """
    assert set(predictions.keys()) == set(ground_truths.keys()), \
        "predictions và ground_truths phải có cùng set image_id"

    scores = {}

    # ---------- BLEU 1~4 ----------
    bleu_scorer = Bleu(4)
    bleu_scores, _ = bleu_scorer.compute_score(ground_truths, predictions)
    for i, score in enumerate(bleu_scores):
        scores[f"BLEU-{i+1}"] = round(score, 4)

    # ---------- CIDEr ----------
    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(ground_truths, predictions)
    scores["CIDEr"] = round(cider_score, 4)

    # ---------- METEOR ----------
    meteor_scorer = Meteor()
    meteor_score, _ = meteor_scorer.compute_score(ground_truths, predictions)
    scores["METEOR"] = round(meteor_score, 4)

    if verbose:
        print("=" * 35)
        print(f"  {'Metric':<12} {'Score':>10}")
        print("=" * 35)
        for name, val in scores.items():
            print(f"  {name:<12} {val:>10.4f}")
        print("=" * 35)

    return scores

# **TRAIN MODEL GỐC**

In [89]:
best_valid_loss = float("inf")
for epoch in range(num_epochs):
    
    model.train()
    train_loss = 0
    for batch in tqdm(train_dataset_loader, "Training", total=len(train_dataset_loader), leave=False):   
      if (current_step + 1) % len(train_dataset_loader) == 0 and current_step!= 0:
        print()
        print(f"Validation at step {current_step}...")
        print()
        # set the model to evaluation mode
        model.eval()
        # initialize our lists that store the predictions and the labels
        predictions, labels = [], []
        # initialize the validation loss
        valid_loss = 0
        for batch in tqdm(valid_dataset_loader):
            # get the batch
            pixel_values = batch["pixel_values"]
            label_ids = batch["labels"]
            # forward pass
            outputs = model(pixel_values=pixel_values, labels=label_ids, output_hidden_states = True)
            loss = outputs.loss# + alpha * int_loss
            valid_loss += loss.item()
            logits = outputs.logits.detach().cpu()
            # add the predictions to the list
            predictions.extend(logits.argmax(dim=-1).tolist())
            # add the labels to the list
            labels.extend(label_ids.tolist())
        eval_prediction = EvalPrediction(predictions=predictions, label_ids=labels)
        # metrics = compute_metrics(eval_prediction)
        print(f"Epoch: {epoch}, Step: {current_step}, Train Loss: {train_loss / len(train_dataset_loader):.4f}, " + 
              f"Valid Loss: {valid_loss / n_valid_steps:.4f}")
        print()
        # save the model
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            print(f"New best model! Valid Loss = {valid_loss:.4f}")
            model.save_pretrained(f"./image-captioning")
            tokenizer.save_pretrained(f"./image-captioning")
            image_processor.save_pretrained(f"./image-captioning")
        # get the model back to train mode
        model.train()
        # reset the train and valid loss
        train_loss, valid_loss = 0, 0
      ### training code below ###
      pixel_values = batch["pixel_values"]
      labels = batch["labels"]
      # forward pass
      outputs = model(pixel_values=pixel_values, labels=labels, output_hidden_states = True)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      loss_v = loss.item()
      train_loss += loss_v
      current_step += 1

KeyboardInterrupt: 

# **TEST Baseline gốc**

In [49]:
import re
id = 0
preds = {}
gts = {}
for item in tqdm(test_samples):
    image = load_image(item['image_path'])
    pixel = image_processor(image, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        predict = model.generate(pixel, max_length = 32)
    cap_predict = tokenizer.batch_decode(predict[0], skip_special_tokens=True)
    cap_predict = re.sub(r'\s+', ' ', cap_predict[0]).strip()
    preds[id] = [cap_predict]
    gts[id] = item['caption']
    id+=1 

100%|██████████| 5/5 [00:02<00:00,  1.96it/s]


In [51]:
scores = compute_caption_metrics(preds, gts)

{'testlen': 130, 'reflen': 77, 'guess': [130, 125, 120, 115], 'correct': [16, 0, 0, 0]}
ratio: 1.6883116882897622
  Metric            Score
  BLEU-1           0.1231
  BLEU-2           0.0000
  BLEU-3           0.0000
  BLEU-4           0.0000
  CIDEr            0.0017
  METEOR           0.0700


In [52]:
scores

{'BLEU-1': 0.1231,
 'BLEU-2': 0.0,
 'BLEU-3': 0.0,
 'BLEU-4': 0.0,
 'CIDEr': np.float64(0.0017),
 'METEOR': 0.07}

# ************Early EXit + KD**
Cần load đúng đường dẫn checkpoint


In [53]:
import torch.nn as nn


**Load lại model teacher**

In [ ]:
##########
'''
Cần load đúng đường dẫn checkpoint
'''
best_model = VisionEncoderDecoderModel.from_pretrained(
    "./image-captioning"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "./image-captioning"
)

image_processor = AutoImageProcessor.from_pretrained(
    "./image-captioning"
)

In [54]:
num_epochs = 5 # number of epochs
batch_size = 4 # the size of batches
current_step = 0
class IntermediateHead(nn.Module):
    def __init__(self, input_size, output_size):
        super(IntermediateHead, self).__init__()
        self.fc = nn.Linear(input_size, output_size)
        
    def forward(self, x):
        return self.fc(x)
    
# Create intermediate head modules
num_intermediate_layers = 12
layers_for_exit = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
intermediate_heads = nn.ModuleList([IntermediateHead(decoder_config.hidden_size, vocab_size) for _ in range(len(layers_for_exit))])

optimizer = AdamW(intermediate_heads.parameters(), lr=1e-4)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=1000, num_training_steps=n_train_steps)

kl_div_loss = torch.nn.KLDivLoss(reduction='batchmean')

# **Train EXIT**

In [88]:
current_step = 0
best_loss = float("inf")
for epoch in range(num_epochs):
    # set the model to training model
    intermediate_heads.train()
    # initialize the training loss
    train_loss = 0
    for batch in tqdm(train_dataset_loader, "Training", total=len(train_dataset_loader), leave=False):
      # train_loss = 0    
      if (current_step +1 ) % len(train_dataset_loader) == 0:
        # evaluate on the validation set
        print()
        print(f"Validation at step {current_step}...")
        print()
        # set the model to evaluation mode
        intermediate_heads.eval()
        # initialize our lists that store the predictions and the labels
        predictions, labels = [], []
        # initialize the validation loss
        valid_loss = 0
        for batch in valid_dataset_loader:
            # get the batch
            pixel_values = batch["pixel_values"]
            label_ids = batch["labels"]
            # forward pass
             #####best_model
            outputs = best_model(pixel_values=pixel_values, labels=label_ids, output_hidden_states = True)
            int_loss = 0
            for exit in range(len(layers_for_exit)):
                intermediate_head = intermediate_heads[exit]
                intermediate_head = intermediate_head.to(device)
                intermediate_logits = intermediate_head(outputs.decoder_hidden_states[layers_for_exit[exit]])
                intermediate_loss = F.cross_entropy(intermediate_logits.view(-1, intermediate_logits.size(-1)), label_ids.view(-1))
                int_loss+=(layers_for_exit[exit]+1)*intermediate_loss
            # get the loss
            loss = (int_loss)
            valid_loss += loss.item()
        if best_loss > (valid_loss/ len(valid_dataset_loader)):
            print(f"Save Epoch: {epoch}, Step: {current_step}, Best Loss: {best_loss:.4f}") 
            intermediate_head_weights_dir = f"./checkpoint/intermediate_head_weights/"
            os.makedirs(intermediate_head_weights_dir, exist_ok=True)

        # Save the weights of each intermediate head
            for layer_idx, intermediate_head in enumerate(intermediate_heads):
                head_path = os.path.join(intermediate_head_weights_dir, f"head_layer_{layers_for_exit[layer_idx]}.pt")
                torch.save(intermediate_head.state_dict(), head_path)
            best_loss = (valid_loss/ len(valid_dataset_loader))
          # print the stats
        print()
        print(f"Epoch: {epoch}, Step: {current_step}, Train Loss: {train_loss/len(train_dataset_loader):.4f}, " + 
              f"Valid Loss: {valid_loss/len(valid_dataset_loader):.4f}")
        print()
        intermediate_head.train()
        train_loss, valid_loss = 0, 0
      #training
      pixel_values = batch["pixel_values"]
      labels = batch["labels"]
      # forward pass
        #####best_model
      outputs = best_model(pixel_values=pixel_values, labels=labels, output_hidden_states = True)
      int_loss_train = 0
      # w = [i / num_intermediate_layers for i in range(num_intermediate_layers)]
      for exit in range(len(layers_for_exit)):
          intermediate_head = intermediate_heads[exit]
          intermediate_head = intermediate_head.to(device)
          intermediate_logits = intermediate_head(outputs.decoder_hidden_states[layers_for_exit[exit]])
          compare_head = intermediate_heads[-1]
          compare_head = compare_head.to(device)
          compare_head_logits = compare_head(outputs.decoder_hidden_states[-1])
          kl_loss = kl_div_loss(F.log_softmax(intermediate_logits, dim=-1), F.softmax(outputs.logits, dim=-1))
          intermediate_loss = (F.cross_entropy(intermediate_logits.view(-1, intermediate_logits.size(-1)), labels.view(-1)))# + 0.8*cosine_loss
          int_loss_train+=0.5*intermediate_loss + 0.5*kl_loss #+ (current_step / n_train_steps)*cosine_loss_11
          # Backpropagate the intermediate loss and accumulate gradients
      # get the loss
      int_loss_train = int_loss_train / len(layers_for_exit)
      # backward pass
      int_loss_train.backward()
      # update the weights
      optimizer.step()
      scheduler.step()
      # zero the gradients
      optimizer.zero_grad()
      loss_v = int_loss_train.item()
      train_loss += loss_v
      # increment the step
      current_step += 1
      
      

KeyboardInterrupt: 

# Infers
Load đúng đường dẫn

In [ ]:
for layer_idx, intermediate_head in enumerate(intermediate_heads):
    head_path = os.path.join(
        intermediate_head_weights_dir,
        f"head_layer_{layers_for_exit[layer_idx]}.pt"
    )

    state_dict = torch.load(head_path, map_location=device)
    intermediate_head.load_state_dict(state_dict)
    intermediate_head.to(device)
    intermediate_head.eval()

In [85]:
predictions = {}
layer_list = []
threshold = 1.5
id = 0
with torch.no_grad():

    for item in tqdm(test_samples):

        image = load_image(item['image_path'])
        pixel_values = image_processor(image, return_tensors="pt").pixel_values.to(device)
        decoder_input_ids = torch.tensor(
            [[tokenizer.bos_token_id]],
            device=device
        )

        preds = []

        for step in range(max_length):

            outputs = model(
                pixel_values=pixel_values,
                decoder_input_ids=decoder_input_ids,
                output_hidden_states=True
            )

            final_logits = outputs.logits

            chosen_token = None
            chosen_layer = None

            prev_token = None
            agg_conf = 0

            # kiểm tra các exit layer
            for exit_idx, intermediate_head in enumerate(intermediate_heads):

                hidden = outputs.decoder_hidden_states[
                    layers_for_exit[exit_idx]
                ]

                logits = intermediate_head(hidden)

                token = logits[:, -1, :].argmax(-1).item()

                probs = torch.softmax(
                    logits[:, -1, :],
                    dim=-1
                )

                conf = probs.max(-1).values.item()

                if prev_token is None:
                    agg_conf = conf

                elif prev_token == token:
                    agg_conf += conf

                else:
                    agg_conf = conf

                prev_token = token

                if agg_conf >= threshold:

                    chosen_token = token
                    chosen_layer = exit_idx
                    break

            # không layer nào đủ confidence
            if chosen_token is None:

                chosen_token = (
                    final_logits[:, -1, :]
                    .argmax(-1)
                    .item()
                )

                chosen_layer = len(layers_for_exit)

            preds.append(chosen_token)
            layer_list.append(chosen_layer)

            # EOS
            if chosen_token == tokenizer.eos_token_id:
                break

            next_token = torch.tensor(
                [[chosen_token]],
                device=device
            )

            decoder_input_ids = torch.cat(
                [decoder_input_ids, next_token],
                dim=1
            )
        cap_predict = tokenizer.batch_decode(preds, skip_special_tokens=True)
        cap_predict = re.sub(r'\s+', ' ', cap_predict[0]).strip()
        predictions[id] = [cap_predict]
        id+=1 
        

100%|██████████| 5/5 [00:10<00:00,  2.15s/it]


In [86]:
predictions

{0: ['The first thing to do is to look at the history of the term "gay" in the United States. The term was coined by the late, great'],
 1: ['The U.S. Department of Justice has charged a former FBI agent with conspiracy to commit wire fraud and other charges in connection with the 2016 election.'],
 2: ['The U.S. Department of Justice has charged a former FBI agent with conspiracy to commit wire fraud and money laundering. The indictment alleges that Robert'],
 3: ['The first time I saw the first time I saw the first time I saw the first time I saw the first time I saw the first time I saw the'],
 4: ['The U.S. Department of Justice has filed a lawsuit against the National Security Agency (NSA) over its surveillance programs, saying the agency violated the Fourth']}

**Đánh giá Model có EarlyExit+KD**

In [87]:
scores_exit = compute_caption_metrics(predictions, gts)

{'testlen': 134, 'reflen': 77, 'guess': [134, 129, 124, 119], 'correct': [16, 0, 0, 0]}
ratio: 1.7402597402371394
  Metric            Score
  BLEU-1           0.1194
  BLEU-2           0.0000
  BLEU-3           0.0000
  BLEU-4           0.0000
  CIDEr            0.0012
  METEOR           0.0692


In [ ]:
scores_exit